###Notebook to quickly test the bot###
A hybrid chatbot that can answer from local documents (PDF, DOCX, TXT)
or from Wikipedia. 
Uses 
1) HuggingFace embeddings,
2) ChromaDB for local vector search, and
3) Gradio for an interactive user interface.

Author: Virendra Pratap Singh
License: MIT

In [43]:
import os, re, time, shutil, hashlib
import wikipedia
import gradio as gr
import docx2txt
import PyPDF2
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
from langchain.embeddings.base import Embeddings

In [44]:
class HuggingFaceEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True)
    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True)[0]

embedding_function = HuggingFaceEmbeddings()

In [45]:
DATA_PATH = os.getenv("CHATBOT_DATA_PATH", "datafiles")  # change to location where data files reside 
PERSIST_DIR = "chroma_db"

In [46]:
from chromadb import PersistentClient

def chunk_text(text, size=250, overlap=50):
    # Breaks long text into small overlapping windows
    return [text[i:i+size] for i in range(0, len(text), size - overlap)]

def build_chroma_from_folder(folder, persist_dir, embedding_function):
    print(f"Indexing local files from: {folder}")
    client = PersistentClient(path=persist_dir)
    collection = client.get_or_create_collection(name="local_docs")

    for fn in os.listdir(folder):
        path = os.path.join(folder, fn)
        if not os.path.isfile(path): 
            continue

        if fn.lower().endswith(".docx"):
            text = docx2txt.process(path)
        elif fn.lower().endswith(".pdf"):
            with open(path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                text = "\n".join(p.extract_text() or "" for p in reader.pages)
        elif fn.lower().endswith(".txt"):
            with open(path, encoding="utf-8") as f:
                text = f.read()
        else:
            continue

        for chunk in chunk_text(text):
            emb = embedding_function.embed_query(chunk).tolist()
            collection.add(
                documents=[chunk],
                embeddings=[emb],
                metadatas=[{"source": fn}],
                ids=[f"{fn}_{hash(chunk)}"]
            )

    print("Rebuilt ChromaDB with chunked documents.")
    return collection



# --- Build or rebuild the collection safely ---
try:
    # If there's an old Chroma client open, close it first
    import gc
    gc.collect()
    time.sleep(1)
    
    if os.path.exists(PERSIST_DIR):
        print("Cleaning up old Chroma directory...")
        # Rename before deletion to avoid Windows locking issues
        temp_dir = PERSIST_DIR + "_old"
        os.rename(PERSIST_DIR, temp_dir)
        shutil.rmtree(temp_dir, ignore_errors=True)

except PermissionError:
    print("Some files were locked. Skipping delete and reusing existing DB.")

os.makedirs(PERSIST_DIR, exist_ok=True)
vectorstore = build_chroma_from_folder(DATA_PATH, PERSIST_DIR, embedding_function)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Add of existing embedding ID: Canada.docx_794513159814758364
Insert of existing embedding ID: Canada.docx_794513159814758364
Add of existing embedding ID: Canada.docx_5103032978245352824
Insert of existing embedding ID: Canada.docx_5103032978245352824


Cleaning up old Chroma directory...
Some files were locked. Skipping delete and reusing existing DB.
Indexing local files from: C:\Users\vps\Documents\AI_ML_Learn\chatbot\datafiles


Add of existing embedding ID: Senegal.docx_5163908369028519835
Insert of existing embedding ID: Senegal.docx_5163908369028519835
Add of existing embedding ID: Senegal.docx_-510830327383004013
Insert of existing embedding ID: Senegal.docx_-510830327383004013


Rebuilt ChromaDB with chunked documents.


In [47]:
import re
import numpy as np
import wikipediaapi

# ---------- Wikipedia Setup ----------
wiki_api = wikipediaapi.Wikipedia(
    language="en",
    user_agent="LocalWikiChatbot/1.0 (https://github.com/yourname; contact@example.com)"
)

# ---------- Wikipedia Query ----------
def query_wiki(question):
    try:
        # Clean up question to extract key term
        key = re.sub(r"(?i)\b(what is|who is|where is|tell me about|describe)\b", "", question).strip()
        if not key:
            key = question.strip()
        key = key.title()

        page = wiki_api.page(key)
        if not page.exists():
            # Fallback search
            import wikipedia
            results = wikipedia.search(key)
            if results:
                page = wiki_api.page(results[0])
            else:
                return f"Wikipedia\n No matching page found for '{key}'."

        summary = page.summary[:1000]
        return f"**Wikipedia**\n {page.title}\n\n{summary}\n\n Source: {page.fullurl}"

    except Exception as e:
        return f"Wiki error: {e}"

# ---------- Local Query ----------
def query_local(question, threshold=0.25):
    #the threshold could be made a parameter
    if not vectorstore:
        return "!!! No local database available."

    try:
        # Embed the query and query local Chroma collection
        query_vec = embedding_function.embed_query(question).tolist()
        res = vectorstore.query(
            query_embeddings=[query_vec],
            n_results=2,
            include=["documents", "metadatas", "distances"],
        )

        if not res or not res.get("documents") or not res["documents"][0]:
            return "!!! No relevant local documents found."

        doc_text = res["documents"][0][0]
        doc_meta = res["metadatas"][0][0]
        score = float(res["distances"][0][0]) if res.get("distances") else 1.0
        similarity = 1 - score  # Convert distance to similarity

        if similarity < threshold:
            return f"!!! Local docs had weak matches (similarity={similarity:.2f})."

        src = doc_meta.get("source", "unknown")
        return f" **{src}**  _(similarity={similarity:.2f})_\n\n{doc_text[:600]}..."

    except Exception as e:
        return f"Local search error: {e}"

# ---------- Decision Logic ----------
def get_response(query, mode):
    """
    mode = "local"  :  use local files first, fallback to wiki only if no local hit
    mode = "wiki"   :  use only Wikipedia, skip local search entirely
    """
    if mode == "wiki":
        return query_wiki(query)

    # local-first fallback
    local_resp = query_local(query)
    if local_resp.startswith("!!!"):
        wiki_resp = query_wiki(query)
        return wiki_resp
    return local_resp


In [48]:
def chat_interface(query, mode):
    return get_response(query, mode)

demo = gr.Interface(
    fn=chat_interface,
    inputs=[
        gr.Textbox(label="Ask me something"),
        gr.Radio(choices=["local", "wiki"], value="local", label="Mode")
    ],
    outputs="markdown",
    title="Smart Chatbot – Local + Wikipedia",
    description="Ask general or document-based questions. Local mode falls back to Wikipedia when needed."
)

demo.launch(debug=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
